# Optimize the 2050 energy system with varying constraints on burden shifts

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
import pickle

In [3]:
save_results = True
run_model = False
update_dat_files = False
limited_ccs = True
run_soo = True

In [4]:
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '00_Shared'))

In [5]:
from utils import (
    run_opti,
    add_biogenic_climate_change_to_impact_scores_df,
    add_rhhd_and_reqd_to_impact_scores_df,
    get_impact_scores,
    category_to_sector,
    aggregate_mobility_submodels,
    plot_configuration_sector,
    plot_contribution_by_sector,
    rename_bio_resources,
    update_ampl_files,
    update_existing_infrastructure_metrics,
    techs_color_map,
    wood_list, wet_biomass_list, waste_list,
    LCA_DATA_FILES_DIR,
    AMPL_FILES_DIR,
    COMMON_DATA_DIR,
    REF_RESULTS,
    N_capita_2050,
    run_order_burden_shifts,
    default_colors_sankey,
    plot_sankey_carbon_flows,
)
from new_plots import _create_sankey_figure, generate_sankey_flows
import bw2data as bd
import pandas as pd
from mescal import Database
import ast

In [6]:
path_lca_data_2050 = AMPL_FILES_DIR / 'data' / '2050'
results_folder = f"{'SOO' if run_soo else '2050'}{'_limited_CCS' if limited_ccs else ''}"

In [7]:
impact_abbrev = pd.read_csv(COMMON_DATA_DIR / 'impact_abbrev.csv')
impact_abbrev.Impact_category = impact_abbrev.Impact_category.apply(lambda x: ast.literal_eval(x))
energyscope_model = pd.read_csv(COMMON_DATA_DIR / 'model_2050.csv')
es_tech_df = pd.read_csv(COMMON_DATA_DIR / 'technology_dictionary.csv')

In [8]:
es_tech_df = es_tech_df[~es_tech_df['Programming name'].isin(wood_list+wet_biomass_list+waste_list)]  # keeping ES names for aggregation of biomass resources later on
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))

## Define scenarios and run the model

In [9]:
policies_scenarios = {
    'NZ All': {'limit_rhh': True, 'limit_req': True, 'limit_acc': True, 'nz': 'energy'},
    'NZ CCA REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': True, 'nz': 'energy'},
    'NZ CCA RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': True, 'nz': 'energy'},
    'NZ CCA': {'limit_rhh': False, 'limit_req': False, 'limit_acc': True, 'nz': 'energy'},
    'NZ RHH REQ': {'limit_rhh': True, 'limit_req': True, 'limit_acc': False, 'nz': 'energy'},
    'NZ REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': False, 'nz': 'energy'},
    'NZ RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': False, 'nz': 'energy'},
    'NZ None': {'limit_rhh': False, 'limit_req': False, 'limit_acc': False, 'nz': 'energy'},

    'NN All': {'limit_rhh': True, 'limit_req': True, 'limit_acc': True, 'nz': 'economy'},
    'NN CCA REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': True, 'nz': 'economy'},
    'NN CCA RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': True, 'nz': 'economy'},
    'NN CCA': {'limit_rhh': False, 'limit_req': False, 'limit_acc': True, 'nz': 'economy'},
    'NN RHH REQ': {'limit_rhh': True, 'limit_req': True, 'limit_acc': False, 'nz': 'economy'},
    'NN REQ': {'limit_rhh': False, 'limit_req': True, 'limit_acc': False, 'nz': 'economy'},
    'NN RHH': {'limit_rhh': True, 'limit_req': False, 'limit_acc': False, 'nz': 'economy'},
    'NN None': {'limit_rhh': False, 'limit_req': False, 'limit_acc': False, 'nz': 'economy'},
}

In [10]:
if run_soo:
    policies_scenarios = {'NZ All': {'limit_rhh': True, 'limit_req': True, 'limit_acc': True, 'nz': 'energy'}}
    obj_list = ['TotalCost', 'TotalLCIA_REQD', 'TotalLCIA_RHHD', 'TotalABROAD_m_CCS_all']
else:
    obj_list = ['TotalCost']

In [11]:
background_scenarios_list = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [12]:
ssp_rcp_emissions_grouping = {
    'low': ['SSP1-L', 'SSP2-PkBudg1000', 'SSP2-RCP26'],
    'medium': ['SSP2-M', 'SSP2-NPi', 'SSP2-RCP45'],
    'high': ['SSP3-H', 'SSP3-rollBack', 'SSP2-Base'],
}
ssp_rcp_emissions_grouping_rev = {v[i]: k for k, v in ssp_rcp_emissions_grouping.items() for i in range(len(v))}

In [13]:
bd.projects.set_current('ecoinvent3.12-backup')

In [14]:
if update_dat_files:
    for bg_scen in background_scenarios_list:
        db = Database([
            f'ecoinvent_cutoff_3.12_{bg_scen["model"]}_{bg_scen["pathway"]}_2050 - regionalized',
            f'EnergyScope_CA-QC_{bg_scen["model"]}_{bg_scen["pathway"]}_2050',
        ])
        update_ampl_files(
            iam_scenarios = [bg_scen],
            year = 2050,
            specific_lcia_abbrev = ['RHHD', 'REQD', 'm_CCS_all'],
            main_database = db,
            direct_emissions_files = False,
            territorial_emissions_files = True,
            ecoinvent_version = '3.12',
            iw_version = '2.2.1',
        )

In [15]:
results_dict = {}

if run_model:

    for obj in obj_list:

        if obj not in results_dict:
            results_dict[obj] = {}

        with open(AMPL_FILES_DIR / 'model' / 'QC_objective_function.mod', 'w') as f:
            f.write(f'drop obj;\nminimize obj2: sum{{y in YEARS}} {obj}[y];')

        for policy_scenario, constraints in policies_scenarios.items():

            if policy_scenario not in results_dict[obj]:
                results_dict[obj][policy_scenario] = {}

            for bg_scenario in background_scenarios_list:

                model = bg_scenario['model']
                pathway = bg_scenario['pathway']
                if model not in results_dict[obj][policy_scenario]:
                    results_dict[obj][policy_scenario][model] = {}
                if pathway not in results_dict[obj][policy_scenario][model]:
                    results_dict[obj][policy_scenario][model][pathway] = {}

                if obj == 'TotalCost' or not run_soo:
                    constraint_total_cost = None
                else:
                    constraint_total_cost = 1.1 * results_dict['TotalCost'][policy_scenario][model][pathway]['TotalCost']

                results_dict[obj][policy_scenario][model][pathway]['results'] = run_opti(
                    iam_scenario=bg_scenario,
                    constraint_on_remaining_hh=constraints['limit_rhh'],
                    constraint_on_remaining_eq=constraints['limit_req'],
                    constraint_on_foreign_ghg_emissions=constraints['limit_acc'],
                    constraint_on_territorial_ghg_emissions=constraints['nz'],
                    constraint_on_total_cost=constraint_total_cost,
                )

                results_dict[obj][policy_scenario][model][pathway]['TotalCost'] = results_dict[obj][policy_scenario][model][pathway]['results'].variables['TotalCost']['TotalCost'].iloc[0]

                if save_results:
                    with open(f"../03_Results/Pickle/{results_folder}/results_{obj}_{model}_{pathway}_{policy_scenario.replace(' ', '_')}.pkl", 'wb') as f:
                        pickle.dump(results_dict[obj][policy_scenario][model][pathway]['results'], f)

else:
    # To skip the optimization part
    for obj in obj_list:
        results_dict[obj] = {}
        for policy_scenario, constraints in policies_scenarios.items():
            results_dict[obj][policy_scenario] = {}
            for bg_scenario in background_scenarios_list:
                model = bg_scenario['model']
                pathway = bg_scenario['pathway']
                with open(f"../03_Results/Pickle/{results_folder}/results_{obj}_{model}_{pathway}_{policy_scenario.replace(' ', '_')}.pkl", 'rb') as f:
                    if model not in results_dict[obj][policy_scenario]:
                        results_dict[obj][policy_scenario][model] = {}
                    if pathway not in results_dict[obj][policy_scenario][model]:
                        results_dict[obj][policy_scenario][model][pathway] = {}
                    results_dict[obj][policy_scenario][model][pathway]['results'] = pickle.load(f)

## Extract results

In [16]:
impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12')
    | (i[0] == 'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)')
]

impact_categories_list +=[
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Total human health (biogenic)'),
    ('IMPACT World+ Midpoint 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)', 'Midpoint', 'Climate change, short term, total'),
]

impact_categories_list = [i for i in impact_categories_list if i[2] not in ['Total ecosystem quality', 'Total human health']]
impact_categories_list = [i for i in impact_categories_list if  # keep only climate change and marine acidification from -1/+1 version of IW+
    not (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12' and ('Marine acidification' in i[2] or 'Climate change' in i[2]))
]

impact_categories_list = [i for i in impact_categories_list if not ('Climate change' in i[2] and 'total' not in i[2])]  # keep only climate change total

In [17]:
imp_cat_code_dict = {
    'Remaining ecosystem quality': 'REQ',
    'Remaining human health': 'RHH',
    'Total ecosystem quality': 'TTEQ',
    'Total human health': 'TTHH',
    'Total ecosystem quality (biogenic)': 'TTEQ',
    'Total human health (biogenic)': 'TTHH',
    'Climate change, short term, total (territorial)': 'CCT',
    'Climate change, short term, total (abroad)': 'CCA',
    'Climate change, short term, total': 'CC',
}

In [ ]:
limits = {}
impact_scores_2023 = pd.read_csv(LCA_DATA_FILES_DIR / '2023' / 'impact_scores.csv')
df_contrib_ccst_2023 = pd.read_csv(LCA_DATA_FILES_DIR / '2023' / 'contribution_analysis_all_processes_ccst.csv')

for bg_scenario in background_scenarios_list:

    model = bg_scenario['model']
    pathway = bg_scenario['pathway']

    impact_scores = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'impact_scores.csv')
    impact_scores_direct = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'impact_scores_direct_emissions.csv')
    df_contrib_ccst = pd.read_csv(LCA_DATA_FILES_DIR / '2050' / model / pathway / 'contribution_analysis_all_processes_ccst.csv')
    norm_factors = pd.read_csv(path_lca_data_2050 / model / pathway / 'QC_techs_lca_max.csv')
    max_ccs_norm = norm_factors.set_index('Abbrev').loc['m_CCS_all']['max_unit']

    impact_scores = update_existing_infrastructure_metrics(
        df_impact_2050=impact_scores,
        df_impact_2020=impact_scores_2023,
        list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
    )

    df_contrib_ccst = update_existing_infrastructure_metrics(
        df_impact_2050=df_contrib_ccst,
        df_impact_2020=df_contrib_ccst_2023,
        list_existing_techs=['HYDRO_DAM', 'HYDRO_RIVER', 'WIND_ONSHORE'],
        col_name='act_name',
        col_type='act_type',
    )

    if model not in limits:
        limits[model] = {}
    limits[model][pathway] = {
        'RHH': results_dict['TotalCost']['NZ All'][model][pathway]['results'].parameters['limit_lcia'].loc['RHHD']['limit_lcia'],
        'REQ': results_dict['TotalCost']['NZ All'][model][pathway]['results'].parameters['limit_lcia'].loc['REQD']['limit_lcia'],
        'CCA': results_dict['TotalCost']['NZ All'][model][pathway]['results'].parameters['limit_abroad'].loc['m_CCS_all']['limit_abroad'],
    }

    limits[model][pathway]['CCA'] *= norm_factors.set_index('Abbrev').loc['m_CCS_all']['max_unit']
    limits[model][pathway]['RHH'] *= norm_factors.set_index('Abbrev').loc['RHHD']['max_unit']
    limits[model][pathway]['REQ'] *= norm_factors.set_index('Abbrev').loc['REQD']['max_unit']

    for policy_scenario in policies_scenarios.keys():

        for obj in obj_list:

            df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
                impact_category=impact_categories_list,
                df_impact_scores=impact_scores,
                df_terr_abroad_ccst=df_contrib_ccst,
                df_results=results_dict[obj][policy_scenario][model][pathway]['results'],
            )

            df_annual_prod_direct = get_impact_scores(
                assessment_type='direct',
                impact_category=impact_categories_list,
                df_impact_scores=impact_scores_direct,
                df_results=results_dict[obj][policy_scenario][model][pathway]['results'],
            )

            df_f_mult = df_f_mult[df_f_mult.F_Mult != 0]
            df_annual_prod = df_annual_prod[df_annual_prod.Annual_Prod != 0]
            df_annual_prod_direct = df_annual_prod_direct[df_annual_prod_direct.Annual_Prod != 0]
            df_annual_res = df_annual_res[df_annual_res.Annual_Res != 0]

            df_f_mult['Objective function'] = obj
            df_annual_prod['Objective function'] = obj
            df_annual_prod_direct['Objective function'] = obj
            df_annual_res['Objective function'] = obj

            df_f_mult['IAM'] = model
            df_annual_prod['IAM'] = model
            df_annual_prod_direct['IAM'] = model
            df_annual_res['IAM'] = model

            df_f_mult['SSP-RCP'] = pathway
            df_annual_prod['SSP-RCP'] = pathway
            df_annual_prod_direct['SSP-RCP'] = pathway
            df_annual_res['SSP-RCP'] = pathway

            df_f_mult['Policy'] = policy_scenario
            df_annual_prod['Policy'] = policy_scenario
            df_annual_prod_direct['Policy'] = policy_scenario
            df_annual_res['Policy'] = policy_scenario

            df_f_mult = df_f_mult.merge(
                energyscope_model[energyscope_model['Amount'] == 1],
                left_on='index',
                right_on='Name',
                how='left',
            ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

            df_annual_prod = df_annual_prod.merge(
                energyscope_model[energyscope_model['Amount'] == 1],
                left_on='index',
                right_on='Name',
                how='left',
            ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

            df_annual_prod_direct = df_annual_prod_direct.merge(
                energyscope_model[energyscope_model['Amount'] == 1],
                left_on='index',
                right_on='Name',
                how='left',
            ).rename(columns={'Flow': 'Main production'}).drop(columns=['Name', 'Amount'])

            df_f_mult['Main production'] = df_f_mult['Main production'].astype(str)
            df_annual_prod['Main production'] = df_annual_prod['Main production'].astype(str)
            df_annual_prod_direct['Main production'] = df_annual_prod_direct['Main production'].astype(str)

            df_annual_prod['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
            df_annual_prod_direct['Sector'] = df_annual_prod.apply(category_to_sector, axis=1)
            df_f_mult['Sector'] = df_f_mult.apply(category_to_sector, axis=1)
            df_annual_res['Sector'] = df_annual_res.apply(
                lambda x: 'Electricity' if x['index'] == 'ELECTRICITY_EHV'
                else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'),
                axis=1)
            df_annual_res['Category'] = df_annual_res.apply(
                lambda x: 'ELECTRICITY_EHV' if x['index'] == 'ELECTRICITY_EHV'
                else ('Biomass' if x['index'] in wood_list+wet_biomass_list+waste_list else 'Imports'),
                axis=1)

            results_dict[obj][policy_scenario][model][pathway]['df_f_mult'] = df_f_mult
            results_dict[obj][policy_scenario][model][pathway]['df_annual_prod'] = df_annual_prod
            results_dict[obj][policy_scenario][model][pathway]['df_annual_prod_direct'] = df_annual_prod_direct
            results_dict[obj][policy_scenario][model][pathway]['df_annual_res'] = df_annual_res

In [18]:
scenarios = [(bg_scenario['model'], bg_scenario['pathway']) for bg_scenario in background_scenarios_list]
scenarios = [(policy, scenario[0], scenario[1]) for policy in policies_scenarios.keys() for scenario in scenarios]
scenarios = [(obj, scenario[0], scenario[1], scenario[2]) for obj in obj_list for scenario in scenarios]

In [ ]:
df_f_mult = pd.concat([results_dict[obj][policy_scenario][model][pathway]['df_f_mult'] for obj, policy_scenario, model, pathway in scenarios])
df_annual_prod = pd.concat([results_dict[obj][policy_scenario][model][pathway]['df_annual_prod'] for obj, policy_scenario, model, pathway in scenarios])
df_annual_prod_direct = pd.concat([results_dict[obj][policy_scenario][model][pathway]['df_annual_prod_direct'] for obj, policy_scenario, model, pathway in scenarios])
df_annual_res = pd.concat([results_dict[obj][policy_scenario][model][pathway]['df_annual_res'] for obj, policy_scenario, model, pathway in scenarios])

In [ ]:
cat_list = [i[-1] for i in impact_categories_list if
            i[1] in ['Human health', 'Ecosystem quality']
            and i[-1] not in ['Total human health', 'Total ecosystem quality']
            ] + ['Climate change, short term, total', 'Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']

df_annual_prod_indirect = df_annual_prod.merge(
    df_annual_prod_direct,
    on=['Objective function', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod'],
    suffixes=('', ' (direct)'),
    how='left',
)
for col in list(df_annual_prod.columns):
    if col not in ['Objective function', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod', 'Phase', 'Main production']:
        if col in ['Climate change, short term, total (territorial)', 'Climate change, short term, total (abroad)']:
            df_annual_prod_direct[col] = 0  # phases are won't be used for these two categories
        else:
            df_annual_prod_indirect[col] = df_annual_prod_indirect[col] - df_annual_prod_indirect[f"{col} (direct)"]
            df_annual_prod_indirect.drop(columns=[f"{col} (direct)"], inplace=True)

df_annual_prod_indirect = df_annual_prod_indirect[['Objective function', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Category', 'Annual_Prod'] + cat_list]

col = ['Objective function', 'Policy', 'IAM', 'SSP-RCP', 'index', 'Sector', 'Phase'] + cat_list
df_f_mult['Phase'] = 'Construction'
df_annual_prod['Phase'] = 'Operation'
df_annual_prod_direct['Phase'] = 'Operation (direct)'
df_annual_prod_indirect['Phase'] = 'Operation (indirect)'
df_annual_res['Phase'] = 'Resource'

df_total_impact = pd.concat([
    df_f_mult[['F_Mult'] + col].rename(columns={'F_Mult': 'Capacity or production'}),
    df_annual_prod_direct[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_prod_indirect[['Annual_Prod'] + col].rename(columns={'Annual_Prod': 'Capacity or production'}),
    df_annual_res[['Annual_Res'] + col].rename(columns={'Annual_Res': 'Capacity or production'}),
],
    ignore_index=True)

In [ ]:
all_mob_techs = (
    list(results_dict['TotalCost']['NZ All']['image']['SSP1-L']['results'].sets['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_FREIGHTMOB_ALL_DISTANCES'])
    + list(results_dict['TotalCost']['NZ All']['image']['SSP1-L']['results'].sets['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PRIVATEMOB_ALL_DISTANCES'])
    + list(results_dict['TotalCost']['NZ All']['image']['SSP1-L']['results'].sets['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES']['TECHNOLOGIES_OF_PUBLICMOB_ALL_DISTANCES'])
)

# Keeping mobility sub-models only for production and impact dataframe (main models are kept if df_f_mult because they have the cost data)
df_total_impact = df_total_impact[~df_total_impact['index'].isin(all_mob_techs)]
df_annual_prod = df_annual_prod[~df_annual_prod['index'].isin(all_mob_techs)]
df_annual_prod_direct = df_annual_prod_direct[~df_annual_prod_direct['index'].isin(all_mob_techs)]

df_total_impact = aggregate_mobility_submodels(df_total_impact)
df_annual_prod = aggregate_mobility_submodels(df_annual_prod)
df_annual_prod_direct = aggregate_mobility_submodels(df_annual_prod_direct)

df_total_impact['index'] = df_total_impact['index'].replace(es_tech_name_dict)
df_annual_prod['index'] = df_annual_prod['index'].replace(es_tech_name_dict)
df_f_mult['index'] = df_f_mult['index'].replace(es_tech_name_dict)
df_annual_res['index'] = df_annual_res['index'].replace(es_tech_name_dict)
df_annual_prod_direct['index'] = df_annual_prod_direct['index'].replace(es_tech_name_dict)

In [19]:
path_results_tables = f"../03_Results/Tables/{results_folder}/"

In [ ]:
if save_results:
    df_f_mult.to_csv(path_results_tables+'df_f_mult.csv', index=False)
    df_annual_prod.to_csv(path_results_tables+'df_annual_prod.csv', index=False)
    df_annual_res.to_csv(path_results_tables+'df_annual_res.csv', index=False)
    df_annual_prod_direct.to_csv(path_results_tables+'df_annual_prod_direct.csv', index=False)
    df_total_impact.to_csv(path_results_tables+'df_total_impact.csv', index=False)

In [20]:
# To skip previous parts (if results are already generated)
# df_f_mult = pd.read_csv(path_results_tables+'df_f_mult.csv')
# df_annual_prod = pd.read_csv(path_results_tables+'df_annual_prod.csv')
# df_annual_res = pd.read_csv(path_results_tables+'df_annual_res.csv')
# df_annual_prod_direct = pd.read_csv(path_results_tables+'df_annual_prod_direct.csv')
# df_total_impact = pd.read_csv(path_results_tables+'df_total_impact.csv')

In [ ]:
active_constraints = {
    'RHH': [],
    'REQ': [],
    'CCA': [],
}

for bg_scenario in background_scenarios_list:
    for policy_scenario in policies_scenarios.keys():
        for cat in ['Remaining ecosystem quality', 'Remaining human health', 'Climate change, short term, total (abroad)']:
            total_impact = df_total_impact[
                (df_total_impact['IAM'] == bg_scenario['model'])
                & (df_total_impact['SSP-RCP'] == bg_scenario['pathway'])
                & (df_total_impact['Policy'] == policy_scenario)
            ][cat].sum()

            limit_impact = limits[bg_scenario['model']][bg_scenario['pathway']][imp_cat_code_dict[cat]]

            if (imp_cat_code_dict[cat] in policy_scenario or policy_scenario == 'All') & (abs((total_impact - limit_impact) / limit_impact) < 1e-5):
                active_constraints[imp_cat_code_dict[cat]].append((bg_scenario['model'], bg_scenario['pathway'], policy_scenario))

In [ ]:
active_constraints

## Plot the results

In [21]:
import plotly.io as pio
import plotly.express as px
import plotly.graph_objs as go
pio.renderers.default = "png"

In [22]:
df_total_impact['Climate change, short term, total'] *= 1e3 / N_capita_2050  # convert to t CO2 eq. per capita and per year
df_total_impact['Climate change, short term, total (abroad)'] *= 1e3 / N_capita_2050
df_total_impact['Climate change, short term, total (territorial)'] *= 1e3 / N_capita_2050

In [23]:
df_total_impact['Total ecosystem quality (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining ecosystem quality'] *= 1e6 / N_capita_2050
df_total_impact['Total human health (biogenic)'] *= 1e6 / N_capita_2050
df_total_impact['Remaining human health'] *= 1e6 / N_capita_2050

In [ ]:
for bg_scenario in background_scenarios_list:
    limits[bg_scenario['model']][bg_scenario['pathway']]['REQ'] *= 1e6 / N_capita_2050
    limits[bg_scenario['model']][bg_scenario['pathway']]['RHH'] *= 1e6 / N_capita_2050
    limits[bg_scenario['model']][bg_scenario['pathway']]['CCA'] *= 1e3 / N_capita_2050

In [24]:
df_total_impact['GHG emissions level'] = df_total_impact.apply(lambda x: ssp_rcp_emissions_grouping_rev[x['SSP-RCP']], axis=1)

### Sankey diagrams of energy flows

In [25]:
for obj, policy_scenario, model, pathway in scenarios:

    df_sankey = generate_sankey_flows(
        results=results_dict[obj][policy_scenario][model][pathway]['results'],
        aggregate_mobility=True,
        aggregate_grid=True,
        aggregate_technology=True,
        run_id=0,
    )
    df_sankey['source (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
    df_sankey['target (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

    fig = _create_sankey_figure(df_sankey, colors=default_colors_sankey, long_names=True)

    if save_results:
        file_path = f'../03_Results/Figures/{results_folder}/{model}/{pathway}/'
        Path(file_path).mkdir(parents=True, exist_ok=True)
        fig.write_html(file_path+f'sankey_{obj}_{model}_{pathway}_{policy_scenario.lower().replace(" ", "_")}.html')

### Sankey of carbon flows

In [26]:
for distance_level in ['_SD', '_MD', '_LD', '_ELD']:
    energyscope_model['Name'] = energyscope_model['Name'].str.replace(distance_level, '')
    energyscope_model['Flow'] = energyscope_model['Flow'].str.replace(distance_level, '')
energyscope_model.drop_duplicates(inplace=True)

energyscope_model['Name'] = energyscope_model['Name'].replace(es_tech_name_dict)

In [29]:
for obj, policy_scenario, model, pathway in scenarios:
    df_plot = df_total_impact[
        (df_total_impact['Objective function'] == obj)
        & (df_total_impact['Policy'] == policy_scenario)
        & (df_total_impact['IAM'] == model)
        & (df_total_impact['SSP-RCP'] == pathway)
    ].rename(columns={'Policy': 'Run'})

    fig = plot_sankey_carbon_flows(
        run=policy_scenario,
        df_total_impact=df_plot,
        model=energyscope_model,
        cutoff=0,
        aggregate_technologies=True,
        show_figure=False,
        return_figure=True,
        save_results=False,
        per_capita=False,
        mode='mfa',
    )

    if save_results:
        file_path = f'../03_Results/Figures/{results_folder}/{model}/{pathway}/'
        fig.write_html(file_path+f'sankey_carbon_{obj}_{model}_{pathway}_{policy_scenario.lower().replace(" ", "_")}.html')

### Parallel coordinates

In [ ]:
str_to_num_ssp_rcp = {
    'low': 0,
    'medium': 1,
    'high': 2,
}

str_to_num_iam = {
    'image': 0,
    'remind': 1,
    'message': 2,
    'tiam-ucl': 3,
}

str_to_num_policy = {key: i for i, key in enumerate(policies_scenarios)}

In [ ]:
def plot_parallel_coord(show_fig: bool=False, save_fig: bool=save_results):

    df_paral_coord = df_total_impact.groupby(['IAM', 'GHG emissions level', 'Policy']).sum().reset_index()
    df_paral_coord['IAM'] = df_paral_coord['IAM'].apply(lambda x: str_to_num_iam)
    df_paral_coord['GHG emissions level'] = df_paral_coord['GHG emissions level'].apply(lambda x: str_to_num_ssp_rcp)
    df_paral_coord['Policy'] = df_paral_coord['Policy'].apply(lambda x: str_to_num_policy)

    fig = px.parallel_coordinates(
        df_paral_coord,
        dimensions=[
            'GHG emissions level',
            'IAM',
            'Policy',
            'Remaining ecosystem quality',
            'Remaining human health',
            'Climate change, short term, total',
        ],
        # color='GHG emissions level',
        # color_continuous_scale=px.colors.cyclical.Phase,
        # color_continuous_midpoint=2,
        # labels={},
    )

    fig.data[0].dimensions[0].update(tickvals=list(str_to_num_ssp_rcp.values()), ticktext=list(str_to_num_ssp_rcp.keys()))
    fig.data[0].dimensions[1].update(tickvals=list(str_to_num_iam.values()), ticktext=list(str_to_num_iam.keys()))
    fig.data[0].dimensions[2].update(tickvals=list(str_to_num_policy.values()), ticktext=list(str_to_num_policy.keys()))

    if show_fig:
        fig.show()

    if save_fig:
        fig.write_html(f'../03_Results/Figures/{results_folder}/parallel_coord.html')

In [ ]:
plot_parallel_coord(False, True)

### Contributions of technologies/sectors

In [ ]:
def plot_contribution_by_sector_burden(
        model,
        pathway,
        impact_category,
        contribution_of: str = 'tech',
        save_fig: bool = save_results,
        show_fig: bool = False,
):

    fig = plot_contribution_by_sector(
        df=df_total_impact[
            (df_total_impact['IAM'] == model)
            & (df_total_impact['SSP-RCP'] == pathway)
        ].rename(columns={
            'Policy': 'Run',
            'Total human health (biogenic)': 'Total human health',
            'Total ecosystem quality (biogenic)': 'Total ecosystem quality',
        }),
        imp_cat=impact_category,
        save_results=False,
        group_by='index' if contribution_of == 'tech' else 'Sector',
        cutoff=0.025 if contribution_of == 'tech' else 0,
        showlegend=True,
        show_direct_marker=False,
        show_total_marker=True,
        show_regionalized_marker=False,
        hatch_phase=False,
        multiple_rcp=False,
        study='burden',
        x_label_name='Scenario',
        return_fig=True,
        show_fig=False,
    )

    if impact_category in [
        'Remaining ecosystem quality',
        'Remaining human health',
        'Climate change, short term, total (abroad)',
    ]:

        fig.add_hline(
            y=limits[model][pathway][imp_cat_code_dict[impact_category]],
            line_width=3,
            line_dash="dash",
            line_color='red',
            annotation_text='No burden shift',
            annotation=dict(font_color='red'),
        )

    if show_fig:
        fig.show()

    if save_fig:
        fig.write_html(f'../03_Results/Figures/{results_folder}/{model}/{pathway}/contrib_{contribution_of}_{model}_{pathway}_{imp_cat_code_dict[impact_category].lower()}.html')

In [ ]:
for cat in [
    'Total ecosystem quality',
    'Total human health',
    'Remaining ecosystem quality',
    'Remaining human health',
    'Climate change, short term, total (abroad)',
    'Climate change, short term, total (territorial)',
    'Climate change, short term, total',
]:
    for contrib_type in ['tech', 'sector']:
        for bg_scenario in background_scenarios_list:
            plot_contribution_by_sector_burden(
                model=bg_scenario['model'],
                pathway=bg_scenario['pathway'],
                impact_category=cat,
                contribution_of=contrib_type,
            )

In [ ]:
plot_contribution_by_sector_burden(
    model='tiam-ucl',
    pathway='SSP2-RCP26',
    impact_category='Remaining human health',
    show_fig=True,
)

### Distribution of impacts across IAMs and SSP-RCP scenarios

In [ ]:
def plot_impact_distribution(impact_category, show_fig: bool=True, save_fig: bool=save_results):

    df_plot = df_total_impact.groupby(['Objective function', 'Policy', 'IAM', 'SSP-RCP'])[[impact_category]].sum().reset_index()
    # df_plot['Constraint'] = df_plot.apply(lambda x: True if (imp_cat_code_dict[impact_category] in x['Policy'] or 'All' in x['Policy']) else False, axis=1)

    fig = px.box(
        df_plot,
        x='Objective function',
        y=impact_category,
        points='all',
        # color='Constraint',
        category_orders={'Policy': run_order_burden_shifts, 'Objective function': obj_list},
        hover_data=['Objective function', 'Policy', 'IAM', 'SSP-RCP'],
    )

    # fig.update_yaxes(range=[0, None])

    if show_fig:
        fig.show()

    if save_fig:
        fig.write_html(f'../03_Results/Figures/{results_folder}/distribution_{imp_cat_code_dict[impact_category]}.html')

In [ ]:
plot_impact_distribution('Remaining human health')

In [ ]:
plot_impact_distribution('Remaining ecosystem quality')

In [ ]:
plot_impact_distribution('Climate change, short term, total (abroad)')

### Influence of IAMs and SSP-RCP scenarios

In [ ]:
df_plot = df_total_impact.groupby(['IAM', 'GHG emissions level', 'Policy']).sum().reset_index()

In [ ]:
fig = px.imshow(
    df_plot.pivot_table(index='Policy', columns='IAM', values='Climate change, short term, total (abroad)', aggfunc='mean')
)

fig.show()

In [ ]:
fig = px.imshow(
    df_plot.pivot_table(index='Policy', columns='GHG emissions level', values='Climate change, short term, total (abroad)', aggfunc='mean')
)

fig.show()

In [ ]:
df_plot_pivot = df_plot.pivot(index='Policy', columns=['IAM', 'SSP-RCP category'], values='Climate change, short term, total (abroad)')
df_plot_pivot.columns = [f"{i[0]} - {i[1]}" for i in df_plot_pivot.columns]

fig = px.imshow(
    df_plot_pivot
)

fig.show()

### Cost

In [ ]:
tc_dict = {}
for obj, policy_scenario, model, pathway in scenarios:
    if obj not in tc_dict:
        tc_dict[obj] = {}
    if policy_scenario not in tc_dict[obj]:
        tc_dict[obj][policy_scenario] = {}
    if model not in tc_dict[obj][policy_scenario]:
        tc_dict[obj][policy_scenario][model] = {}
    tc_dict[obj][policy_scenario][model][pathway] = results_dict[obj][policy_scenario][model][pathway]['results'].variables['TotalCost']['TotalCost'].iloc[0]

In [ ]:
delta_rel_dict = {}
for obj, policy_scenario, model, pathway in scenarios:
    if obj not in delta_rel_dict:
        delta_rel_dict[obj] = {}
    if policy_scenario not in delta_rel_dict[obj]:
        delta_rel_dict[obj][policy_scenario] = {}
    if model not in delta_rel_dict[obj][policy_scenario]:
        delta_rel_dict[obj][policy_scenario][model] = {}

    if run_soo:
        delta_rel_dict[obj][policy_scenario][model][pathway] = 100 * (tc_dict[obj][policy_scenario][model][pathway] - tc_dict['TotalCost'][policy_scenario][model][pathway]) / tc_dict['TotalCost'][policy_scenario][model][pathway]
    else:
        delta_rel_dict[obj][policy_scenario][model][pathway] = 100 * (tc_dict[obj][policy_scenario][model][pathway] - tc_dict[obj]['None'][model][pathway]) / tc_dict[obj]['None'][model][pathway]

In [ ]:
tc_dict

In [ ]:
delta_rel_dict

In [ ]:
def plot_cost_difference(
        df_f_mult: pd.DataFrame,
        df_annual_res: pd.DataFrame,
        cutoff: float = 0,
        relative: bool = True,
        save_results: bool = save_results,
):

    df_f_mult['Cost'] = df_f_mult['C_inv_an'] + df_f_mult['C_maint']
    df_cost = pd.concat([df_f_mult[['Run', 'index', 'Cost']], df_annual_res[['Run', 'index', 'C_op']].rename(columns={'C_op': 'Cost'})])
    df_cost['Cost'] *= 1e6 / N_capita_2050  # from MCAD to CAD/cap

    # Ensure every index in the reference has an entry for every Run in df_cost
    for name in df_cost['index'].unique():
        for run in df_cost['Run'].unique():
            if run != 'None':  # Skip the default run
                if len(df_cost[(df_cost['index'] == name) & (df_cost['Run'] == run)]) == 0:
                    # Add a row with Production=0
                    new_row = {'index': name, 'Run': run, 'Cost': 0}
                    df_cost = pd.concat([df_cost, pd.DataFrame([new_row])], ignore_index=True)

    df_cost = pd.merge(
        df_cost,
        df_cost[df_cost['Run'] == 'None'][['index', 'Cost']],
        on=['index'],
        how='left',
        suffixes=('', '_ref'),
    )

    df_cost['Cost_ref'] = df_cost['Cost_ref'].fillna(0)
    df_cost['Delta'] = df_cost['Cost'] - df_cost['Cost_ref']
    df_cost['Delta_rel'] = 100 * df_cost['Delta'] / df_cost[df_cost['Run'] == 'None']['Cost'].sum()

    # Apply cutoff
    df_cost['index'] = df_cost.apply(lambda x: x['index'] if abs(x['Delta_rel']) >= cutoff else 'Other', axis=1)

    # Sort bars
    df_grouped_index = df_cost.groupby(['Run', 'index'])['Delta'].sum().reset_index()
    index_order = [
                      tec for tec in
                      df_grouped_index[df_grouped_index['Delta'] < 0].sort_values(by='Delta', ascending=False)['index'].unique().tolist() +
                      df_grouped_index[df_grouped_index['Delta'] > 0].sort_values(by='Delta', ascending=False)['index'].unique().tolist()
                      if tec != 'Other'
                  ] + ['Other']

    if relative:
        y_var = 'Delta_rel'
        y_label = 'Relative cost difference with respect to <i>None</i> scenario (%)'
    else:
        y_var = 'Delta'
        y_label = 'Cost difference with respect to <i>None</i> scenario (CAD/cap)'

    fig = px.bar(
        df_cost[df_cost['Run'] != 'None'].groupby(['Run', 'index'])[[y_var]].sum().reset_index(),
        y=y_var,
        x='Run',
        color='index',
        category_orders={'Run': run_order_burden_shifts, 'index': index_order},
        color_discrete_map=techs_color_map,
        labels={
            'Run': 'Scenario',
            y_var: y_label,
            'index': 'Technology or resource',
        }
    )

    fig.update_layout(legend_traceorder="reversed", margin=dict(t=5, b=5, l=5, r=5))

    x = df_cost[df_cost['Run'] != 'None'].groupby(['Run'])[[y_var]].sum().reset_index()['Run']
    y = df_cost[df_cost['Run'] != 'None'].groupby(['Run'])[[y_var]].sum().reset_index()[y_var]

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers",
            marker=dict(color="red", size=8, symbol="circle", ),
            name="Net cost difference",
        )
    )

    fig.show()

    if save_results:
        fig.write_image(f'../03_Results/Figures/cost_difference_{'rel' if relative else 'abs'}.pdf')
        fig.update_layout(width=None, height=None)
        fig.write_html(
            f'../03_Results/Figures/cost_difference_{'rel' if relative else 'abs'}.html',
            include_plotlyjs=True,
            full_html=True,
        )

In [ ]:
plot_cost_difference(
    df_f_mult=df_f_mult,
    df_annual_res=df_annual_res,
    cutoff=1.0, # [%]
    relative=True,
    save_results=save_results,
)

In [ ]:
plot_cost_difference(
    df_f_mult=df_f_mult,
    df_annual_res=df_annual_res,
    cutoff=1.0, # [%]
    relative=False,
    save_results=save_results,
)